# Project: Analysis of Indoor Environmental Sensor Data Using Python
### AF1780 - Data Analysis and Programming for Management and Finance.
#### Oscar Arandes Tejerina

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### 1. Data Understanding and Preparation

The dataset is given in an Excel file containing two sheets, corresponding to the ground floor and first floor, respectively. After loading the data, the “Time stamp” column is converted to a datetime format and set as the DataFrame index. This allows the data to be sorted chronologically, since the original dataset was not ordered in time (the latest records appeared first).

Next, all sensor readings are converted to numeric format, since they were initially stored as "object" types. Then, the data is resampled to a fixed 10-minute interval by computing the mean of all observations within each time window (__resample("10min").mean()__). This produces a consistent and regularly spaced time series.

Additionally, missing values are handled using interpolation (__interpolate(method = "time")__). Since the variables represent continuous physical processes (such as sensor measurements), missing values are estimated based on neighbouring observations. However, interpolation cannot fill missing values at the beginning or end of the dataset, so we apply forward (__ffill()__) and backward (__bfill()__) fill methods in these cases.

Finally, we present basic descriptive statistics for the data.

In [ ]:
# Read the dataset for both the ground and first floors
ground_data = pd.read_excel("data.xlsx", sheet_name = "Ground floor")
first_data  = pd.read_excel("data.xlsx", sheet_name = "First floor")

# Convert the "Time stamp" to datetime 
ground_data["Time stamp"] = pd.to_datetime(ground_data["Time stamp"], dayfirst = True)
first_data["Time stamp"]  = pd.to_datetime(first_data["Time stamp"], dayfirst = True)

# Set the "Time stamp" as Index and Sort the data to be time-ordered 
ground_data = ground_data.set_index("Time stamp").sort_index()
first_data  = first_data.set_index("Time stamp").sort_index()

# Convert the attributes to numeric
ground_data = ground_data.apply(pd.to_numeric, errors = "coerce")
first_data  = first_data.apply(pd.to_numeric, errors = "coerce")

# Resample the data every 10 min
ground_data = ground_data.resample("10min").mean()
first_data  = first_data.resample("10min").mean()

# Handle missing values by interpolation
ground_data = ground_data.interpolate(method = "time").ffill().bfill()
first_data  = first_data.interpolate(method = "time").ffill().bfill()

# Basic Statistics for the datasets
display(ground_data.describe())
display(first_data.describe())

#### 2. Time-Series Visualization

In this section, we present time-series visualizations of the sensor measurements. Specifically, we analyze indoor (room) temperature over time at different time scales (hourly, daily, weekly, and monthly) and compare indoor and outdoor temperatures. We also examine humidity trends across different time scales and explore the relationship between air quality and indoor temperature. In addition, we visualize the heating and cooling setpoints on a single graph and present a three-variable visualization of AHU supply temperature, outdoor temperature, and air quality. All plots are generated separately for the ground floor and the first floor.

In [ ]:
# Grouping data by different periods 
freqs = {
    "Hourly": "1h",
    "Daily": "1D",
    "Weekly": "1W",
    "Monthly": "1ME"
}

ground_resampled = {}
first_resampled = {}

for name, time in freqs.items():
    ground_resampled[name] = ground_data.resample(time).mean()
    first_resampled[name] = first_data.resample(time).mean()

# ---------------- Indoor Temperature Over Time at Different Time Scales ----------------
fig, axs = plt.subplots(2,1, figsize = (10,10))

# Ground Floor
for freq_name in freqs:
    axs[0].plot(
        ground_resampled[freq_name].index,
        ground_resampled[freq_name]["Room Temperature"],
        label = freq_name
    )
    
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Temperature (°C)")
axs[0].set_title("Indoor Temperature Over Time at Different Time Scales (Ground Floor)")
axs[0].legend()

# First Floor
for freq_name in freqs:
    axs[1].plot(
        first_resampled[freq_name].index,
        first_resampled[freq_name]["Room Temperature"],
        label = freq_name
    )

axs[1].set_xlabel("Time")
axs[1].set_ylabel("Temperature (°C)")
axs[1].set_title("Indoor Temperature Over Time at Different Time Scales (First Floor)")
axs[1].legend()

plt.tight_layout()
plt.show()


# ---------------- Outdoor vs Indoor Temperature  ----------------
fig, axs = plt.subplots(2,1, figsize = (10,10))

# Ground Floor
axs[0].scatter(ground_data["Room Temperature"],
               ground_data["Outdoor Temperature"]
              )
 
axs[0].set_title("Outdoor vs Indoor Temperature (Ground Floor)")
axs[0].set_xlabel("Indoor Temperature (°C)")
axs[0].set_ylabel("Outdoor Temperature (°C)")


# First Floor
axs[1].scatter(first_data["Room Temperature"],
               first_data["Outdoor Temperature"]
              )

axs[1].set_title("Outdoor vs Indoor Temperature (First Floor)")
axs[1].set_xlabel("Indoor Temperature (°C)")
axs[1].set_ylabel("Outdoor Temperature (°C)")

plt.tight_layout()
plt.show()


# ---------------- Humidity Over Time at Different Time Scales ----------------
fig, axs = plt.subplots(2, 1, figsize=(10,10))

# Ground Floor
for freq_name in freqs:
    axs[0].plot(
        ground_resampled[freq_name].index,
        ground_resampled[freq_name]["Humidity"],
        label = freq_name
    )

axs[0].set_title("Humidity Over Time at Different Time Scales (Ground Floor)")
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Relative Humidity (%)")
axs[0].legend()

# First Floor
for freq_name in freqs:
    axs[1].plot(
        first_resampled[freq_name].index,
        first_resampled[freq_name]["Humidity"],
        label = freq_name
    )
    
axs[1].set_title("Humidity Over Time at Different Time Scales (First Floor)")
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Relative Humidity (%)")
axs[1].legend()

plt.tight_layout()
plt.show()


# ---------------- Air Quality vs Indoor Temperature at Different Time Scales ----------------
fig, axs = plt.subplots(2,1, figsize = (10,10))

# Ground Floor
for freq_name in freqs:
    axs[0].scatter(
        ground_resampled[freq_name]["Room Temperature"],
        ground_resampled[freq_name]["Air quality [ppm]"],
        label = freq_name
    )

axs[0].set_title("Air Quality vs Indoor Temperature at Different Time Scales (Ground Floor)")
axs[0].set_xlabel("Indoor Temperature (°C)")
axs[0].set_ylabel("Air Quality (ppm)")
axs[0].legend()

# First Floor
for freq_name in freqs:
    axs[1].scatter(
        first_resampled[freq_name]["Room Temperature"],
        first_resampled[freq_name]["Air quality [ppm]"],
        label = freq_name
    )

axs[1].set_title("Air Quality vs Indoor Temperature at Different Time Scales (First Floor)")
axs[1].set_xlabel("Indoor Temperature (°C)")
axs[1].set_ylabel("Air Quality (ppm)")
axs[1].legend()

plt.tight_layout()
plt.show()


# ---------------- Cooling & Heating Setpoints Over Time ----------------
fig, axs = plt.subplots(2, 1, figsize=(12,13))

# Ground Floor: Cooling setpoint
axs[0].plot(ground_data.index,
            ground_data["SetPoint Temp Cool"],
            label = "Cooling"
           )
    
# Ground Floor: Heating setpoint
axs[0].plot(ground_data.index,
            ground_data["SetPoint Temp Heat"],
            label = "Heating"
           )

axs[0].set_title("Cooling & Heating Setpoints Over Time (Ground Floor)")
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Temperature (°C)")
axs[0].legend()

# First Floor: Cooling setpoint
axs[1].plot(first_data.index,
            first_data["SetPoint Temp Cool"],
            label = "Cooling"
           )
    
# First Floor: Heating setpoint
axs[1].plot(first_data.index,
            first_data["SetPoint Temp Heat"],
            label = "Heating"
           )

axs[1].set_title("Cooling & Heating Setpoints Over Time (First Floor)")
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Temperature (°C)")
axs[1].legend()

plt.tight_layout()
plt.show()


# ---------------- AHU supply temperature vs outdoor temperature vs air quality (in 3d plot) ---------------
fig, axs = plt.subplots(2, 1, figsize=(10, 12))

# Ground Floor
scatter_ground = axs[0].scatter(
    ground_data["Air quality [ppm]"],
    ground_data["Outdoor Temperature"],
    c = ground_data["AHU Supply Temperature"],
    cmap = "viridis"
)

axs[0].set_title("Ground Floor")
axs[0].set_xlabel("Air Quality (ppm)")
axs[0].set_ylabel("Outdoor Temperature (°C)")

cbar1 = plt.colorbar(scatter_ground, ax = axs[0])
cbar1.set_label("AHU Supply Temperature (°C)")


# First Floor
scatter_first = axs[1].scatter(
    first_data["Air quality [ppm]"],
    first_data["Outdoor Temperature"],
    c = first_data["AHU Supply Temperature"],
    cmap = "viridis"
)

axs[1].set_title("First Floor")
axs[1].set_xlabel("Air Quality (ppm)")
axs[1].set_ylabel("Outdoor Temperature (°C)")

cbar2 = plt.colorbar(scatter_first, ax = axs[1])
cbar2.set_label("AHU Supply Temperature (°C)")

fig.suptitle(
    "AHU Supply Temperature vs Outdoor Temperature vs Air Quality",
    fontsize=14
)

plt.tight_layout()
plt.show()


#### 3. Relationship Between Variables

In this section, we study the relationships between different sensor measurements. In particular, we examine the relationship between outdoor and indoor temperature, the relationship between humidity and air quality, and the relationship between setpoints and indoor conditions, including both air quality and indoor temperature. For this, we compute the Pearson correlation coefficient (PCC) for each pair of variables.

In [ ]:
# Calculating Pearson Correlation Coefficients (PCCs) 
corr_outdoor_indoor_temp_ground         = ground_data["Outdoor Temperature"].corr(ground_data["Room Temperature"])
corr_humidity_airQuality_ground         = ground_data["Humidity"].corr(ground_data["Air quality [ppm]"])
corr_indoorTemp_coolingSetpoint_ground  = ground_data["Room Temperature"].corr(ground_data["SetPoint Temp Cool"])
corr_indoorTemp_heatingSetpoint_ground  = ground_data["Room Temperature"].corr(ground_data["SetPoint Temp Heat"])
corr_airQuality_coolingSetpoint_ground  = ground_data["Air quality [ppm]"].corr(ground_data["SetPoint Temp Cool"])
corr_airQuality_heatingSetpoint_ground  = ground_data["Air quality [ppm]"].corr(ground_data["SetPoint Temp Heat"])

corr_outdoor_indoor_temp_first         = first_data["Outdoor Temperature"].corr(first_data["Room Temperature"])
corr_humidity_airQuality_first         = first_data["Humidity"].corr(first_data["Air quality [ppm]"])
corr_indoorTemp_coolingSetpoint_first  = first_data["Room Temperature"].corr(first_data["SetPoint Temp Cool"])
corr_indoorTemp_heatingSetpoint_first  = first_data["Room Temperature"].corr(first_data["SetPoint Temp Heat"])
corr_airQuality_coolingSetpoint_first  = first_data["Air quality [ppm]"].corr(first_data["SetPoint Temp Cool"])
corr_airQuality_heatingSetpoint_first  = first_data["Air quality [ppm]"].corr(first_data["SetPoint Temp Heat"])

# Storing PCCs in a Dictionary
ground_corr = {
    "Outdoor vs Indoor (Room) Temp": corr_outdoor_indoor_temp_ground,
    "Humidity vs Air Quality": corr_humidity_airQuality_ground,
    "Indoor (Room) Temp vs Cooling Setpoint": corr_indoorTemp_coolingSetpoint_ground,
    "Indoor (Room) Temp vs Heating Setpoint": corr_indoorTemp_heatingSetpoint_ground,
    "Air Quality vs Cooling Setpoint": corr_airQuality_coolingSetpoint_ground,
    "Air Quality vs Heating Setpoint": corr_airQuality_heatingSetpoint_ground
}

first_corr = {
    "Outdoor vs Indoor (Room) Temp": corr_outdoor_indoor_temp_first,
    "Humidity vs Air Quality": corr_humidity_airQuality_first,
    "Indoor (Room) Temp vs Cooling Setpoint": corr_indoorTemp_coolingSetpoint_first,
    "Indoor (Room) Temp vs Heating Setpoint": corr_indoorTemp_heatingSetpoint_first,
    "Air Quality vs Cooling Setpoint": corr_airQuality_coolingSetpoint_first,
    "Air Quality vs Heating Setpoint": corr_airQuality_heatingSetpoint_first
}

# Convert to DataFrames
ground_corr_table = pd.DataFrame.from_dict(
    ground_corr,
    orient = "index",
    columns = ["Ground Floor PCC"]
)

first_corr_table = pd.DataFrame.from_dict(
    first_corr,
    orient = "index",
    columns = ["First Floor PCC"]
)

# Display the Results 
results_correlations = pd.concat([ground_corr_table, first_corr_table], axis=1)
display(results_correlations)

# Visualization of the Correlations via Scatter Plots
fig, axs = plt.subplots (6, 2, figsize=(10,20))

# Row 1: Outdoor vs Indoor Temp 
axs[0,0].scatter(ground_data["Outdoor Temperature"], ground_data["Room Temperature"])
axs[0,0].set_title("Outdoor vs Indoor Temperature (Ground)")
axs[0,0].set_xlabel("Outdoor Temperature (°C)")
axs[0,0].set_ylabel("Indoor Temperature (°C)")

axs[0,1].scatter(first_data["Outdoor Temperature"], first_data["Room Temperature"])
axs[0,1].set_title("Outdoor vs Indoor Temperature (First)")
axs[0,1].set_xlabel("Outdoor Temperature (°C)")
axs[0,1].set_ylabel("Indoor Temperature (°C)")

# Row 2: Humidity vs Air Quality 
axs[1,0].scatter(ground_data["Humidity"], ground_data["Air quality [ppm]"])
axs[1,0].set_title("Humidity vs Air Quality (Ground)")
axs[1,0].set_xlabel("Humidity (%)")
axs[1,0].set_ylabel("Air Quality (ppm)")

axs[1,1].scatter(first_data["Humidity"], first_data["Air quality [ppm]"])
axs[1,1].set_title("Humidity vs Air Quality (First)")
axs[1,1].set_xlabel("Humidity (%)")
axs[1,1].set_ylabel("Air Quality (ppm)")

# Row 3: Indoor Temp vs Cooling Setpoint 
axs[2,0].scatter(ground_data["Room Temperature"], ground_data["SetPoint Temp Cool"])
axs[2,0].set_title("Indoor Temp vs Cooling Setpoint (Ground)")
axs[2,0].set_xlabel("Indoor Temperature (°C)")
axs[2,0].set_ylabel("Cooling Setpoint (°C)")

axs[2,1].scatter(first_data["Room Temperature"], first_data["SetPoint Temp Cool"])
axs[2,1].set_title("Indoor Temp vs Cooling Setpoint (First)")
axs[2,1].set_xlabel("Indoor Temperature (°C)")
axs[2,1].set_ylabel("Cooling Setpoint (°C)")

# Row 4: Indoor Temp vs Heating Setpoint 
axs[3,0].scatter(ground_data["Room Temperature"], ground_data["SetPoint Temp Heat"])
axs[3,0].set_title("Indoor Temp vs Heating Setpoint (Ground)")
axs[3,0].set_xlabel("Indoor Temperature (°C)")
axs[3,0].set_ylabel("Heating Setpoint (°C)")

axs[3,1].scatter(first_data["Room Temperature"], first_data["SetPoint Temp Heat"])
axs[3,1].set_title("Indoor Temp vs Heating Setpoint (First)")
axs[3,1].set_xlabel("Indoor Temperature (°C)")
axs[3,1].set_ylabel("Heating Setpoint (°C)")

# Row 5: Air Quality vs Cooling Setpoint 
axs[4,0].scatter(ground_data["Air quality [ppm]"], ground_data["SetPoint Temp Cool"])
axs[4,0].set_title("Air Quality vs Cooling Setpoint (Ground)")
axs[4,0].set_xlabel("Air Quality (ppm)")
axs[4,0].set_ylabel("Cooling Setpoint (°C)")

axs[4,1].scatter(first_data["Air quality [ppm]"], first_data["SetPoint Temp Cool"])
axs[4,1].set_title("Air Quality vs Cooling Setpoint (First)")
axs[4,1].set_xlabel("Air Quality (ppm)")
axs[4,1].set_ylabel("Cooling Setpoint (°C)")

# Row 6: Air Quality vs Heating Setpoint
axs[5,0].scatter(ground_data["Air quality [ppm]"], ground_data["SetPoint Temp Heat"])
axs[5,0].set_title("Air Quality vs Heating Setpoint (Ground)")
axs[5,0].set_xlabel("Air Quality (ppm)")
axs[5,0].set_ylabel("Heating Setpoint (°C)")

axs[5,1].scatter(first_data["Air quality [ppm]"], first_data["SetPoint Temp Heat"])
axs[5,1].set_title("Air Quality vs Heating Setpoint (First)")
axs[5,1].set_xlabel("Air Quality (ppm)")
axs[5,1].set_ylabel("Heating Setpoint (°C)")
        
plt.tight_layout()
plt.show()


__Conclusions__: The correlation analysis shows that most relationships between indoor air quality, humidity, and HVAC setpoints are weak, indicating limited direct linear dependence. The strongest relationship is observed between outdoor and indoor temperature, with a moderate-to-strong positive correlation in both floors (0.69–0.78), suggesting that indoor temperature is influenced by external conditions despite AHU regulation. Moderate negative correlations are observed between indoor temperature and heating setpoints (more notably in the first floor).

#### 4. Setpoint and Comfort Analysis

In this section, we investigate whether indoor conditions remain close to the desired heating and cooling setpoints. To do this, we compute the temperature error relative to the relevant setpoint. During the cold season (October-April), the error is calculated as the difference between the heating setpoint and the indoor temperature. During the hot season (May-September), the error is calculated as the difference between the cooling setpoint and the indoor temperature.

Next, we compute significant deviations by identifying outliers in the temperature error. This is done by selecting points that lie below the first quartile minus 1.5 times the interquartile range (IQR), or above the third quartile plus 1.5 times the IQR. These thresholds allow us to detect unusually large deviations from the expected temperature behavior.

In [ ]:
# Cold season: October-April
cold_months = [10, 11, 12, 1, 2, 3, 4]

# ---------------- Calculating error ----------------
ground_data["Temperature Error"] = np.where(
    ground_data.index.month.isin(cold_months),
    ground_data["SetPoint Temp Heat"] - ground_data["Room Temperature"],
    ground_data["SetPoint Temp Cool"] - ground_data["Room Temperature"]
)

first_data["Temperature Error"] = np.where(
    first_data.index.month.isin(cold_months),
    first_data["SetPoint Temp Heat"] - first_data["Room Temperature"],
    first_data["SetPoint Temp Cool"] - first_data["Room Temperature"]
)

# ---------------- Calculating Significant Deviations ----------------
err_ground = ground_data["Temperature Error"]
err_first  = first_data["Temperature Error"]

Q1g = err_ground.quantile(0.25)
Q1f = err_first.quantile(0.25)
Q3g = err_ground.quantile(0.75)
Q3f = err_first.quantile(0.75)

IQRg = Q3g - Q1g
IQRf = Q3f - Q1f

outliers_ground = (err_ground < Q1g - 1.5 * IQRg) | (err_ground > Q3g + 1.5 * IQRg)
outliers_first  = (err_first  < Q1f - 1.5 * IQRf) | (err_first  > Q3f + 1.5 * IQRf)


# ---------------- Plotting Significant Deviations ----------------
fig, axs = plt.subplots(2,1, figsize=(12,10))

# Ground Floor
axs[0].plot(ground_data.index,
            ground_data["Temperature Error"], 
            label = "Temperature Error")

axs[0].scatter(
    ground_data.index[outliers_ground],
    ground_data["Temperature Error"][outliers_ground],
    color = "red",
    label = "Large deviations"
)

axs[0].axhline(0, 
            color = "black", 
            linestyle = "--")

axs[0].set_title("Temperature Error and Significant Deviations Over Time (Ground Floor)")
axs[0].set_ylabel("Error (°C)")
axs[0].legend()

# First Floor
axs[1].plot(first_data.index, 
            first_data["Temperature Error"], 
            label = "Temperature Error")

axs[1].scatter(
    first_data.index[outliers_first],
    first_data["Temperature Error"][outliers_first],
    color = "red",
    label = "Large deviations"
)

axs[1].axhline(0, 
            color = "black", 
            linestyle = "--")

axs[1].set_title("Temperature Error and Significant Deviations Over Time (First Floor)")
axs[1].set_ylabel("Error (°C)")
axs[1].legend()

plt.tight_layout()
plt.show()

# ---------------- Monthly Count of Large Deviations ----------------
monthly_deviations = pd.DataFrame({
    "Ground Floor": ground_data[outliers_ground].resample("ME").size(),
    "First Floor":  first_data[outliers_first].resample("ME").size()
})

display(monthly_deviations.fillna(""))


# ---------------- Statistical Comparison Between Floors ----------------
ground_deviation_rate = outliers_ground.mean() * 100
first_floor_deviation_rate = outliers_first.mean() * 100

results = pd.DataFrame({
    "Ground Floor": [
        outliers_ground.mean() * 100,
        ground_data["Room Temperature"].std()],
    "First Floor": [
        outliers_first.mean() * 100,
        first_data["Room Temperature"].std()]}, 
    index=["Percentage Large Deviations (%)", "Fluctuations Indoor (Room) Temp (std)"
])

display(results)

__Conclusions__: Based on the monthly count of significantly deviating temperature readings we can extract some conclusions. For the first floor, the largest concentration of deviations occurs in early 2021, particularly in January (4,436 cases) and May (4,229 cases), followed by a gradual reduction throughout the rest of 2021. After mid-2021, deviations become rare and largely disappear until a new increase appears again in 2024–2025, with a notable spike in January 2025 (1,534 cases). Overall, the first floor shows fewer but more clustered periods of extreme deviation.

In contrast, the ground floor exhibits a more scattered pattern over time, with generally lower monthly counts but more intermittent occurrences spread across multiple years. Notable peaks appear in July 2023 (915 cases), August 2023 (525 cases), and December 2023 (135 cases), as well as smaller but recurring deviations in other months.

#### 5. Floor Comparison

In this final section, we compare the two floors based on the analyses presented above. Specifically, we investigate which floor has the higher average temperature, which experiences greater temperature fluctuations, and which exhibits larger comfort deviations, measured as the error relative to the setpoint discussed previously.

In [ ]:
# ---------------- Plot Mean + STD  ----------------
floors = ["Ground Floor", "First Floor"]

means = [ground_data["Room Temperature"].mean(), first_data["Room Temperature"].mean()]
stds = [ground_data["Room Temperature"].std(), first_data["Room Temperature"].std()]

plt.figure(figsize=(7,5))

plt.errorbar(
    floors,
    means,
    yerr = stds,
    fmt = 'o',          
    capsize = 8,
    markersize = 8,
    color = "red",
    ecolor = "green"
)

plt.title("Indoor (Room) Temperature: Mean ± Std by Floor")
plt.ylabel("Temperature (°C)")
plt.xlim(-0.3, 1.3)

plt.grid(axis="y", alpha=0.5)
plt.tight_layout()
plt.show()

__Conclusions__: First, we observe that the first floor has a slightly higher average temperature (19.6 ± 3.2 °C) compared to the ground floor (19.12 ± 3.2 °C). These values can be seen in the table from Section 1 or in the figure above. Both floors exhibit similar temperature fluctuations, with a standard deviation of approximately 3.2 °C in both cases. 

With respect to which floor has the highest comfort fluctuation, we refer to the table in the previous section. We observe that the first floor exhibits a significantly higher number of periods where indoor conditions deviate from the heating and cooling setpoints, with slightly over 8% of readings compared to 1.5% on the ground floor. In this regard, we conclude that the ground floor provides better comfort conditions than the first floor.

From the previous section, we also observed that the first floor experiences more intense but concentrated periods of discomfort, while the ground floor shows more sporadic and distributed deviations. This suggests different thermal behavior: the first floor appears more sensitive to specific periods, whereas the ground floor exhibits more irregular but less extreme deviations over time.